# EXP-029 · Soft visual prefixes for Qwen

Select an **NVIDIA L4**. Confirm the original EXP-028 handoff path, authorize the inner pilot below, and choose **Run all**. Use Technical only to stop after the six-painting gradient check. Outputs and completed checkpoints persist in Drive. This is a research pilot on released training data.

In [ ]:
# OWNER_CONFIGURATION
AUTHORIZE_EXPERIMENT = False
TECHNICAL_ONLY = False
EXECUTION_SCOPE = "RESEARCH_ONLY_CV"  # Ends after this inner pilot. No outer CV.
RESOURCE_ELIGIBILITY_STATUS = "PINNED_RESEARCH_LICENSE_ONLY"
AUTHORIZE_GUARDED_CONFIRMATION = False
REQUIRE_SUBMISSION_ZIP = False
EVAL_INPUT_CSV = None
EVAL_DATA_DIR = None
MAX_GPU_HOURS = 1.5
MAX_TOTAL_WALL_HOURS = 1.5
AUTO_BROWSER_DOWNLOAD = True
AUTO_RELEASE_COLAB = True
HANDOFF_PATH = "/content/drive/MyDrive/exp023_handoff.zip"
PERSISTENT_ROOT = "/content/drive/MyDrive/exp029"
SOURCE = "061f6bf0d3aa4d0b24bfbf1766e9139f307a9d0c"


In [ ]:
import json
import math
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
from datetime import datetime, timezone
from uuid import uuid4

REPOSITORY = "https://github.com/bqa100507-spec/SemEval-2027-Task-12---ArtEmotion-28-Subtask-3---Image-Only-Cross-Cultural-Sentiment-Prediction.git"

def run_cloud():
    if AUTHORIZE_EXPERIMENT is not True:
        print("GPU authorization is OFF. Review the pilot scope before enabling it.")
        return

    from google.colab import drive, files, runtime

    started = time.monotonic()
    local = Path(tempfile.mkdtemp(prefix="exp029-", dir="/content"))
    output = local / "results"
    output.mkdir()
    attempt = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
    destination = Path(PERSISTENT_ROOT) / "attempts" / attempt
    source = local / "source"
    report = {"status": "BLOCKED", "source": SOURCE, "scope": EXECUTION_SCOPE, "attempt": attempt}
    finalize = None
    stage = "CONFIGURATION"
    reserve = 150
    env = dict(os.environ, OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", OMP_THREAD_LIMIT="1",
               MKL_NUM_THREADS="1", NUMEXPR_NUM_THREADS="1", BLIS_NUM_THREADS="1",
               CUBLAS_WORKSPACE_CONFIG=":4096:8", HF_HUB_DISABLE_XET="1")
    env.update(UV_PYTHON_INSTALL_DIR=str(local / "python"), UV_CACHE_DIR=str(local / "uv_cache"))
    env.pop("PYTHONPATH", None)

    def remaining():
        return deadline - time.monotonic() - reserve

    def run(command, name, *, cwd=None):
        nonlocal stage
        stage = name
        allowed = remaining()
        if allowed <= 0:
            raise TimeoutError("Time budget exhausted before " + name)
        print(f"{name} (remaining work budget {allowed:.0f}s)", flush=True)
        with (output / (name + ".log")).open("w") as log:
            result = subprocess.run(command, cwd=cwd, env=env, stdout=log, stderr=subprocess.STDOUT,
                                    text=True, timeout=allowed)
        if result.returncode:
            tail = (output / (name + ".log")).read_text(errors="replace").splitlines()[-15:]
            raise RuntimeError(f"{name} exited with {result.returncode}. " + "\n".join(tail))

    try:
        if (EXECUTION_SCOPE != "RESEARCH_ONLY_CV" or AUTHORIZE_GUARDED_CONFIRMATION or REQUIRE_SUBMISSION_ZIP
                or EVAL_INPUT_CSV is not None or EVAL_DATA_DIR is not None
                or RESOURCE_ELIGIBILITY_STATUS != "PINNED_RESEARCH_LICENSE_ONLY"):
            raise ValueError("EXP-029 permits only training-data research and no confirmation or submission.")
        if any(not math.isfinite(x) or x <= 0 or x > 1.5 for x in (MAX_GPU_HOURS, MAX_TOTAL_WALL_HOURS)):
            raise ValueError("Set finite limits within the 1.5-hour prototype cap.")
        deadline = started + 3600 * min(MAX_GPU_HOURS, MAX_TOTAL_WALL_HOURS)
        stage = "DRIVE_MOUNT"
        drive.mount("/content/drive")
        destination.mkdir(parents=True, exist_ok=False)
        run(["git", "clone", "--no-checkout", REPOSITORY, str(source)], "clone")
        run(["git", "checkout", "--detach", SOURCE], "checkout", cwd=source)
        sys.path.insert(0, str(source / "src"))
        from st3.experiment_workflow import finalize
        stage = "INPUT_AND_GPU"
        if not Path(HANDOFF_PATH).is_file():
            raise FileNotFoundError("Set HANDOFF_PATH to the original EXP-028 exp023_handoff.zip in Drive.")
        run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"], "gpu")
        if "L4" not in (output / "gpu.log").read_text():
            raise RuntimeError("Select an NVIDIA L4 runtime before Run all.")
        print((output / "gpu.log").read_text())

        uv = [sys.executable, "-m", "uv"]
        run([sys.executable, "-m", "pip", "install", "uv==0.7.13"], "uv")
        run([*uv, "python", "install", "3.12.10"], "python")
        run([*uv, "venv", "--python", "3.12.10", str(local / "venv")], "venv")
        python = str(local / "venv/bin/python")
        run([*uv, "pip", "sync", "--python", python, "--extra-index-url",
             "https://download.pytorch.org/whl/cu126", "--index-strategy", "unsafe-best-match",
             "requirements-exp026.lock.txt"], "dependencies", cwd=source)
        run([*uv, "pip", "check", "--python", python], "dependency_check")
        run([python, "-m", "pytest", "-q", "tests/test_exp029.py", "tests/test_experiment_workflow.py"],
            "cpu_tests", cwd=source)

        root = Path(PERSISTENT_ROOT)
        command = [python, "scripts/run_exp029.py", "--source", SOURCE, "--authorize", "RESEARCH_ONLY_L4",
                   "--handoff", HANDOFF_PATH, "--model-cache", str(root / "qwen_assets"),
                   "--models", str(root / "models"), "--output", str(output),
                   "--seconds", str(remaining() + 90)]
        run(command + (["--technical"] if TECHNICAL_ONLY else []), "worker", cwd=source)
        report.update(json.loads((output / "status.json").read_text()))
        print(json.dumps(report, indent=2))
    except Exception as exc:
        failed = output / "status.json"
        if failed.is_file():
            report.update(json.loads(failed.read_text()))
        report.update(stage=stage, attempt_error=f"{type(exc).__name__}: {exc}")
        if isinstance(exc, (TimeoutError, subprocess.TimeoutExpired)):
            report["status"] = "STOPPED_BUDGET"
        print(report)
        raise
    finally:
        report["allocation_elapsed_before_export_seconds"] = time.monotonic() - started
        if finalize is not None:
            finalize(output, destination, report, platform="colab", download=files.download,
                     release=runtime.unassign, auto_download=AUTO_BROWSER_DOWNLOAD, auto_release=AUTO_RELEASE_COLAB)
        else:
            # Preserve bootstrap logs even if the source was unavailable.
            try:
                destination.mkdir(parents=True, exist_ok=True)
                (output / "bootstrap_status.json").write_text(json.dumps(report, indent=2))
                archive = Path(shutil.make_archive(str(local / "outputs"), "zip", output))
                import hashlib
                target = destination / "outputs.zip"
                shutil.copyfile(archive, target)
                digest = hashlib.sha256(archive.read_bytes()).hexdigest()
                if hashlib.sha256(target.read_bytes()).hexdigest() != digest:
                    raise OSError("Bootstrap archive Drive read-back mismatch")
                (destination / "outputs.sha256").write_text(digest + "  outputs.zip\n")
                if AUTO_BROWSER_DOWNLOAD:
                    files.download(str(target))
            finally:
                if AUTO_RELEASE_COLAB:
                    runtime.unassign()

run_cloud()
